# 06 — Bias Correction and Multi-Model Ensemble Analysis

## Part 1 — Research Foundation and Dataset Preparation

This notebook is the sixth stage of the climate attribution repository.

The preceding notebooks established the workflow for:

- reading and validating climate NetCDF data
- preprocessing temperature and precipitation variables
- calculating climate extremes and ETCCDI-style indicators
- analysing temporal trends and regional behaviour

This notebook introduces the next research stage: **bias correction and multi-model ensemble analysis**.

The objective is not simply to make model output resemble observations. A scientifically defensible workflow must distinguish between:

1. systematic model bias,
2. model-to-model structural differences,
3. scenario uncertainty,
4. internal climate variability, and
5. the climate-change signal that should be retained.

Part 1 establishes the reproducible data and validation framework used by the later bias-correction and ensemble sections.


## 1. Research Questions

The workflow addresses four main questions:

1. How well does each climate model reproduce the reference climate?
2. What systematic biases exist in the model simulations?
3. Does statistical bias correction improve agreement with the reference dataset?
4. How can several models be combined while retaining information about model spread and uncertainty?

The later notebook parts will answer these questions quantitatively using distributional, temporal, and spatial diagnostics.


## 2. Scientific Scope

Bias correction is performed on a **model variable or climate indicator** after ensuring that the input datasets are physically meaningful, spatially compatible, and temporally aligned.

Typical variables in this repository include:

- `tasmax` — daily maximum near-surface air temperature
- `tasmin` — daily minimum near-surface air temperature
- `pr` — precipitation
- derived indicators such as TXx, RX1day, RX5day, CDD, R95p, and WSDI

The same framework can be applied to monthly or annual climate indicators when their statistical characteristics are appropriate for the selected correction method.


## 3. Bias Correction: Core Concept

A climate model can reproduce the broad spatial and temporal pattern of climate while still exhibiting a systematic bias.

For a variable \(X\), a simple additive mean bias can be written as:

\[
B = \overline{X}_{model} - \overline{X}_{ref}
\]

where:

- \(X_{model}\) is the model simulation,
- \(X_{ref}\) is the observational/reference dataset,
- \(B\) is the estimated bias.

For temperature, an additive correction is often physically interpretable.

For precipitation, multiplicative or distribution-based methods are often more appropriate because precipitation is non-negative and strongly skewed.

The important principle is that the correction method must be consistent with the variable, temporal resolution, calibration period, and research question.


## 4. Multi-Model Ensembles

A multi-model ensemble does not imply that all models are equally correct.

A basic ensemble mean is:

\[
X_{ens} = \frac{1}{N}\sum_{i=1}^{N}X_i
\]

where \(N\) is the number of models.

The ensemble mean provides a useful central estimate, but it must be accompanied by measures of model spread such as:

- standard deviation,
- inter-model range,
- interquartile range,
- percentile envelopes.

Later sections will also examine model agreement and weighted ensemble approaches.


## 5. Notebook Learning Objectives

By the end of Part 1, you will be able to:

- configure a reproducible climate-analysis environment
- define repository paths without hard-coding machine-specific locations
- discover NetCDF datasets automatically
- inspect dimensions, coordinates, variables, and metadata
- check units and missing values
- identify common data-quality problems before correction
- prepare a consistent model/reference dataset structure

No bias-correction operation is applied until these validation steps pass.


## 6. Required Python Libraries

Core packages:

- `xarray` — labelled multidimensional climate data
- `numpy` — numerical computation
- `pandas` — tabular metadata and diagnostics
- `matplotlib` — visualization
- `scipy` — statistical methods used later

Optional packages can be added later for specialized bias-correction algorithms.

If a package is missing, install it in the project environment rather than modifying the notebook repeatedly.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

from scipy import stats

warnings.filterwarnings("ignore")

print("Core climate-analysis libraries imported successfully.")


## 7. Plotting Configuration

The figures produced in this repository should be readable in both notebooks and exported research reports.

The settings below intentionally avoid hard-coded colour schemes so that individual analyses can choose an appropriate palette when necessary.


In [ ]:
plt.rcParams.update({
    "figure.figsize": (10, 5),
    "figure.dpi": 120,
    "axes.grid": True,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
})

print("Plot configuration ready.")


## 8. Repository Configuration

The repository is expected to contain data, indicators, outputs, notebooks, and supporting scripts.

The paths below are deliberately relative to the project root so that the notebook can be moved between machines.

If your repository uses a different structure, modify only this configuration cell.


In [ ]:
# ------------------------------------------------------------------
# PROJECT PATHS
# ------------------------------------------------------------------

PROJECT_DIR = Path("..").resolve()

DATA_DIR = PROJECT_DIR / "data"
OBS_DIR = DATA_DIR / "observations"
MODEL_DIR = DATA_DIR / "models"

INDICATORS_DIR = PROJECT_DIR / "indicators"
OUTPUT_DIR = PROJECT_DIR / "outputs"

NOTEBOOK_DIR = PROJECT_DIR / "notebooks"

PATHS = {
    "project": PROJECT_DIR,
    "data": DATA_DIR,
    "observations": OBS_DIR,
    "models": MODEL_DIR,
    "indicators": INDICATORS_DIR,
    "outputs": OUTPUT_DIR,
    "notebooks": NOTEBOOK_DIR,
}

for name, path in PATHS.items():
    print(f"{name:15s}: {path}")


## 9. Check Repository Paths

This check does not create directories or modify existing data.

It reports which expected locations are currently available. This is useful when the notebook is opened on a new workstation or server.


In [ ]:
print("Repository path status")
print("-" * 60)

for name, path in PATHS.items():
    status = "FOUND" if path.exists() else "NOT FOUND"
    print(f"{name:15s} {status:10s} {path}")


## 10. NetCDF Dataset Discovery

Climate repositories commonly contain many files distributed across:

- models,
- experiments,
- scenarios,
- variables,
- time periods, and
- processing stages.

Manual file selection is error-prone. The helper below recursively discovers NetCDF files and returns them in deterministic order.


In [ ]:
def discover_netcdf_files(directory):
    """Return NetCDF files below *directory* in deterministic order."""
    directory = Path(directory)

    if not directory.exists():
        return []

    return sorted(directory.rglob("*.nc"))


model_files = discover_netcdf_files(MODEL_DIR)
observation_files = discover_netcdf_files(OBS_DIR)

print(f"Model NetCDF files       : {len(model_files)}")
print(f"Observation NetCDF files : {len(observation_files)}")


## 11. Inspect Discovered Files

The following cell prints a compact inventory.

For a large repository, this is preferable to loading every dataset into memory. File discovery should remain separate from data loading.


In [ ]:
def print_file_inventory(files, title, max_files=20):
    print(f"\n{title}")
    print("=" * len(title))

    if not files:
        print("No NetCDF files found.")
        return

    for file in files[:max_files]:
        print(file)

    if len(files) > max_files:
        print(f"... and {len(files) - max_files} more files.")


print_file_inventory(model_files, "MODEL FILES")
print_file_inventory(observation_files, "REFERENCE / OBSERVATION FILES")


## 12. Dataset Inspection Helper

Before any correction, inspect:

- dimensions,
- coordinate names,
- variable names,
- units,
- calendar information,
- time coverage,
- latitude and longitude ranges.

This prevents a common climate-analysis error: applying a statistically correct method to physically inconsistent datasets.


In [ ]:
def summarize_dataset(ds):
    """Print a compact summary of an xarray Dataset."""

    print("Dimensions")
    print("-" * 60)
    for name, size in ds.sizes.items():
        print(f"{name:15s}: {size}")

    print("\nCoordinates")
    print("-" * 60)
    for name in ds.coords:
        values = ds[name].values
        if values.size:
            print(
                f"{name:15s}: "
                f"dtype={values.dtype}, "
                f"first={values.flat[0]!r}, "
                f"last={values.flat[-1]!r}"
            )

    print("\nData variables")
    print("-" * 60)
    for name, var in ds.data_vars.items():
        print(
            f"{name:15s}: "
            f"dims={var.dims}, "
            f"dtype={var.dtype}, "
            f"units={var.attrs.get('units', 'not specified')}"
        )


## 13. Missing-Value Diagnostics

Missing values are not automatically errors.

They may occur because of:

- land/sea masks,
- invalid observations,
- incomplete time periods,
- quality-control procedures,
- spatial clipping.

The important point is to quantify them before applying interpolation or statistical correction.


In [ ]:
def missing_value_summary(ds):
    """Return missing and valid counts for every data variable."""

    rows = []

    for name, var in ds.data_vars.items():
        total = var.size
        missing = int(var.isnull().sum())
        valid = total - missing

        rows.append({
            "variable": name,
            "total_values": total,
            "valid_values": valid,
            "missing_values": missing,
            "missing_percent": 100 * missing / total if total else np.nan,
        })

    return pd.DataFrame(rows)


# Example:
# missing_value_summary(ds)


## 14. Basic Metadata Validation

The validation function below checks the metadata most important for later processing.

It does not attempt to decide whether a dataset is scientifically correct; it only identifies metadata that should be reviewed.


In [ ]:
def validate_variable_metadata(ds, variables=None):
    """Report basic metadata for selected variables."""

    if variables is None:
        variables = list(ds.data_vars)

    rows = []

    for name in variables:
        if name not in ds:
            rows.append({
                "variable": name,
                "exists": False,
                "units": None,
                "standard_name": None,
                "long_name": None,
            })
            continue

        var = ds[name]

        rows.append({
            "variable": name,
            "exists": True,
            "units": var.attrs.get("units"),
            "standard_name": var.attrs.get("standard_name"),
            "long_name": var.attrs.get("long_name"),
        })

    return pd.DataFrame(rows)


## 15. Safe Dataset Opening

For large climate datasets, eager loading should be avoided.

`xarray.open_dataset()` keeps the data on disk until an operation requires computation. This is especially important for the high-resolution daily datasets used elsewhere in this repository.

The helper below opens a dataset without immediately loading all values into memory.


In [ ]:
def open_climate_dataset(path, chunks=None):
    """Open a NetCDF climate dataset lazily."""

    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"Dataset not found: {path}")

    return xr.open_dataset(path, chunks=chunks)


# Example:
# ds = open_climate_dataset(model_files[0], chunks={"time": 365})


## 16. First Dataset Inspection

If model files were discovered, inspect the first one.

This cell intentionally does not load the complete dataset into memory. The output is used to identify the correct variable and coordinate structure before writing correction code.


In [ ]:
if model_files:
    first_model_file = model_files[0]

    print(f"Opening: {first_model_file}")

    ds_model_example = open_climate_dataset(
        first_model_file,
        chunks={"time": 365},
    )

    summarize_dataset(ds_model_example)
else:
    print("No model NetCDF files are available for inspection.")
    print("Set MODEL_DIR to the correct repository location.")


## 17. Reference Dataset Inspection

Bias correction requires a reference dataset.

The reference may be:

- station observations,
- gridded observations,
- reanalysis,
- a high-resolution observational product,
- or another accepted reference dataset.

The choice of reference dataset should be documented because it directly affects the estimated model bias.


In [ ]:
if observation_files:
    first_observation_file = observation_files[0]

    print(f"Opening: {first_observation_file}")

    ds_obs_example = open_climate_dataset(
        first_observation_file,
        chunks={"time": 365},
    )

    summarize_dataset(ds_obs_example)
else:
    print("No observation/reference NetCDF files are available.")
    print("Set OBS_DIR to the correct repository location.")


## 18. Initial Quality-Control Checklist

Before moving to bias correction, verify:

- [ ] model and reference variables represent the same physical quantity
- [ ] units are compatible
- [ ] calendars are understood
- [ ] time coordinates are monotonic
- [ ] temporal resolution is compatible
- [ ] spatial coordinates are valid
- [ ] longitude conventions are compatible
- [ ] missing values have been quantified
- [ ] the calibration/reference period is explicitly defined
- [ ] no obvious physically impossible values remain

Do not correct a dataset merely because its mean differs from observations. First determine whether the difference is a genuine model bias, a unit problem, a calendar mismatch, a spatial mismatch, or a preprocessing error.


## 19. Part 1 Summary

Part 1 established the research foundation for the bias-correction and ensemble workflow.

We have:

- defined the scientific motivation
- separated model bias from ensemble uncertainty
- configured repository paths
- discovered NetCDF datasets
- created lazy dataset-opening utilities
- established metadata and missing-value diagnostics
- defined the minimum quality-control requirements

### Next

The next part will move from data preparation to **model–reference comparison**, including:

- temporal alignment,
- spatial-grid compatibility,
- calibration periods,
- climatological means,
- seasonal cycles,
- distributions,
- bias maps,
- RMSE,
- MAE, and correlation.
